# 01 · Graph feature extraction

This notebook computes the **27 transparent graph features** of every transaction, in three families:

| Family | Question it answers | Features | Output file |
|---|---|---|---|
| A | How many links does the transaction have? | 8 | `A_features.csv` |
| B | What position does it hold in the money flow of its time step? | 12 | `B_features.csv` |
| C | What shape does its neighbourhood have? | 7 | `C_features.csv` |

**Input** (in `DATA_PATH` on Google Drive): `txs_edgelist.csv` and `txs_features.csv` (only the columns `txId` and `Time step` are read).

**Output** (in `DATA_PATH/01-graph-feature-extractions/`): the three CSV files above. Each file has one row per transaction (203,769 rows, same order as `txs_features.csv`), the key column `txId` and the **raw** feature values. There is no transformation (log, scaling) and no feature selection here: that happens in a later step. `Time step` is not saved, so that no later step can use it as a feature by mistake.

**How to run it.** `Runtime → Run all`. Every result is written to Drive as soon as it is ready:
- if a family file already exists, that family is skipped;
- the slow features (betweenness, closeness, Ollivier–Ricci curvature) save one checkpoint file per time step in `_partial/`. If Colab disconnects, run the notebook again: it restarts from the first missing time step. The `_partial/` folders are deleted when their family file is complete;
- set `FORCE_RECOMPUTE = True` (section 0) to delete the outputs of this notebook and start from scratch.

**Run time.** Family A takes seconds, family C a few minutes. Family B is the slowest part (exact betweenness and closeness on every time step). The log prints the time spent and an estimate of the remaining time.

## 0. Setup

In [1]:
#!rm -rf /content/drive/MyDrive/UNIVERSITA/block-crypto/data/01-graph-feature-extractions

In [2]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('igraph', 'igraph')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [3]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
OUT_DIR = DATA_PATH / '01-graph-feature-extractions'   # final files of this notebook
PARTIAL_DIR = OUT_DIR / '_partial'                      # checkpoints of the slow features
LOCAL_INPUT = Path('/content/input')                    # local copy of the input files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Copy the input files from Drive to the Colab disk: reading a local file is much faster.
# The outputs are never copied: they are always read and written directly on Drive.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for name in ['txs_edgelist.csv', 'txs_features.csv']:
    source, target = DATA_PATH / name, LOCAL_INPUT / name
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{name}: {target.stat().st_size / 1e6:,.0f} MB')

txs_edgelist.csv: 4 MB
txs_features.csv: 695 MB


In [5]:
import math
import os
import shutil
import time
import warnings
from collections import deque
from fractions import Fraction

import igraph as ig
import networkx as nx
import numpy as np
import pandas as pd

# Parameters of the features. They are fixed by definition, not tuned on the data.
PAGERANK_ALPHA = 0.85        # B1, B2: probability of following an edge instead of jumping
KATZ_GAMMA = 0.5             # B10, B11: each extra step of a path weighs half
EIG_ZERO = 1e-12             # B6: smaller eigenvector scores are below the solver precision -> 0
ORC_ALPHA = Fraction(1, 2)   # C6, C7: share of mass that stays on the node (exact fraction)
FAN_SHARE = 0.05             # C1, C2: at most this share of the train nodes is flagged as fan-in / fan-out
TRAIN_STEPS = range(1, 31)   # train time steps (used only for the C1, C2 thresholds; no labels involved)

# True = delete every output of this notebook on Drive and compute everything again.
FORCE_RECOMPUTE = False

if FORCE_RECOMPUTE and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
def write_csv(df, path):
    """Write a CSV safely: first to a temporary file, then rename it.
    If the run stops while writing, no half-written file is left on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False, float_format='%.10g')
    os.replace(tmp, path)


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


class Progress:
    """Prints one line per time step with the time spent and an estimate of the time left.
    The estimate is weighted by the expected work of each time step, not by its number of
    nodes, because a large time step costs much more than a small one."""

    def __init__(self, name, work):
        self.name, self.work = name, work                 # work = {time step: expected cost}
        self.total_work = sum(work.values())
        self.total_nodes = sum(len(STEP_NODES[t]) for t in work)
        self.done_work = self.done_steps = self.done_nodes = 0
        self.start = time.time()

    def step(self, t, seconds):
        self.done_work += self.work[t]
        self.done_steps += 1
        self.done_nodes += len(STEP_NODES[t])
        elapsed = time.time() - self.start
        left = elapsed / self.done_work * (self.total_work - self.done_work)
        print(f'[{self.name}] time step {t:2d} ({len(STEP_NODES[t]):,} nodes) in {fmt_time(seconds)} | '
              f'{self.done_steps}/{len(self.work)} steps, {self.done_nodes:,}/{self.total_nodes:,} nodes, '
              f'{100 * self.done_work / self.total_work:.1f}% of the work | '
              f'elapsed {fmt_time(elapsed)}, left ~{fmt_time(left)}', flush=True)


def run_per_step(name, compute, work):
    """Compute a slow feature group one time step at a time.

    After each time step the result is saved to `_partial/<name>/ts_XX.csv`. Time steps
    that already have a checkpoint are skipped, so an interrupted run resumes where it
    stopped. Returns the results of all time steps, read back from Drive."""
    folder = PARTIAL_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    path = lambda t: folder / f'ts_{t:02d}.csv'
    todo = [t for t in STEPS if not path(t).exists()]
    print(f'[{name}] {len(STEPS) - len(todo)} time steps already on Drive, {len(todo)} to compute.')
    if todo:
        progress = Progress(name, {t: work[t] for t in todo})
        for t in todo:
            start = time.time()
            result = compute(*STEP_GRAPH[t])
            result.insert(0, 'txId', TXID[STEP_NODES[t]])
            write_csv(result, path(t))
            progress.step(t, time.time() - start)
    return pd.concat([pd.read_csv(path(t)) for t in STEPS], ignore_index=True)


def in_node_order(df):
    """Put the rows in the order of txs_features.csv and check that every node appears exactly once."""
    assert df['txId'].is_unique and len(df) == N, 'missing or duplicated nodes'
    df = df.set_index('txId').reindex(TXID)
    assert not df.isna().any().any(), 'missing values'
    return df.reset_index()

## 1. Feature definitions

### Notation

- $G_t = (V_t, E_t)$ is the directed graph of time step $t$, with $n_t = \lvert V_t \rvert$ nodes. An edge $(u, v)$ means that **$v$ spends an output of $u$**: money flows from $u$ to $v$. Every feature of a node is computed **inside its own time step**. There are no edges between time steps and every time step is one connected component (both are checked in section 3), so this is exact.
- $N^-(v) = \{u : (u,v) \in E_t\}$ are the predecessors and $N^+(v) = \{w : (v,w) \in E_t\}$ the successors of $v$. $d^-(v) = \lvert N^-(v) \rvert$ is the in-degree and $d^+(v) = \lvert N^+(v) \rvert$ the out-degree.
- $\vec d(u, v)$ is the length of the shortest **directed** path from $u$ to $v$ ($\infty$ if there is none). $\mathrm{dist}(u, v)$ is the distance in the **undirected** graph $\hat G_t$.
- $N(v) = N^-(v) \cup N^+(v)$ and $\deg(v) = d^-(v) + d^+(v)$ are the neighbours and the degree in $\hat G_t$ (the union is disjoint: there are no reciprocal edges).
- $A$ is the adjacency matrix of $G_t$ ($A_{uv} = 1$ if $(u,v) \in E_t$), $\hat A = A + A^\top$ the undirected one, $\mathbf 1$ the all-ones vector, $\mathbb N = \{0, 1, 2, \dots\}$.
- **No feature uses the labels.** All features are computed on every node (licit, illicit and unknown; train, validation and test) without leakage.

**When a value is divided by the size of the time step.** Only when the raw value sums over *all* nodes or *all* pairs of the time step, because then it grows with $n_t$ by construction (B1, B2, B3, B7, B8, B12; B6 is scaled by its maximum). Counts of things that a node touches or reaches stay raw (A1–A8, B4, B5, B9–B11, C1–C5): a chain of 10 steps is 10 steps long in a small and in a large time step.

### Family A — "How many links does it have?" (8 features)

**A1 · `in_deg`** — domain $\mathbb N$
$$d^-(v)$$
Number of transactions of the dataset whose outputs $v$ spends.

**A2 · `out_deg`** — domain $\mathbb N$
$$d^+(v)$$
Number of transactions of the dataset that spend outputs of $v$.

**A3 · `anc2`** — domain $\{0, \dots, n_t - 1\}$
$$\big\lvert \{u : \vec d(u,v) = 2\} \big\rvert$$
"Grandparents": the history of the money two steps back, without the nodes already one step back.

**A4 · `desc2`** — domain $\{0, \dots, n_t - 1\}$
$$\big\lvert \{w : \vec d(v,w) = 2\} \big\rvert$$
Where the money goes two steps later.

**A5 · `pred_out_mean`** — domain $\{0\} \cup [1, \infty)$
$$\frac{1}{d^-(v)} \sum_{u \in N^-(v)} d^+(u) \qquad (0 \text{ if } d^-(v) = 0)$$
Into how many pieces did my predecessors split their money, on average? A high value means that $v$ comes from a distribution and has many "siblings".

**A6 · `pred_out_max`** — domain $\mathbb N$
$$\max_{u \in N^-(v)} d^+(u) \qquad (0 \text{ if } d^-(v) = 0)$$
Like A5, for the most extreme predecessor.

**A7 · `succ_in_mean`** — domain $\{0\} \cup [1, \infty)$
$$\frac{1}{d^+(v)} \sum_{w \in N^+(v)} d^-(w) \qquad (0 \text{ if } d^+(v) = 0)$$
When the money of $v$ is spent, with how many other transactions is it merged? A high value means that $v$ flows into a consolidation.

**A8 · `succ_in_max`** — domain $\mathbb N$
$$\max_{w \in N^+(v)} d^-(w) \qquad (0 \text{ if } d^+(v) = 0)$$
Like A7, for the most extreme successor.

In [7]:
def neighbour_lists(n, src, dst):
    """Predecessors and successors of every node, as Python lists."""
    preds = [[] for _ in range(n)]
    succs = [[] for _ in range(n)]
    for u, v in zip(src.tolist(), dst.tolist()):
        succs[u].append(v)
        preds[v].append(u)
    return preds, succs


def count_at_distance_two(nbrs):
    """A3 / A4. For every node, the number of nodes at distance exactly 2 along `nbrs`
    (predecessors -> ancestors, successors -> descendants)."""
    out = np.zeros(len(nbrs), dtype=np.int64)
    for v, first in enumerate(nbrs):
        second = set()
        for x in first:
            second.update(nbrs[x])
        second.difference_update(first)   # nodes at distance 1 are not at distance 2
        second.discard(v)
        out[v] = len(second)
    return out


def family_A(n, src, dst):
    in_deg = np.bincount(dst, minlength=n)
    out_deg = np.bincount(src, minlength=n)
    preds, succs = neighbour_lists(n, src, dst)

    # Along every edge u -> v, v looks at the out-degree of u and u looks at the in-degree of v.
    pred_out_max = np.zeros(n, dtype=np.int64)
    np.maximum.at(pred_out_max, dst, out_deg[src])
    succ_in_max = np.zeros(n, dtype=np.int64)
    np.maximum.at(succ_in_max, src, in_deg[dst])
    pred_out_sum = np.bincount(dst, weights=out_deg[src], minlength=n)
    succ_in_sum = np.bincount(src, weights=in_deg[dst], minlength=n)

    return pd.DataFrame({
        'in_deg': in_deg,
        'out_deg': out_deg,
        'anc2': count_at_distance_two(preds),
        'desc2': count_at_distance_two(succs),
        'pred_out_mean': np.divide(pred_out_sum, in_deg, out=np.zeros(n), where=in_deg > 0),
        'pred_out_max': pred_out_max,
        'succ_in_mean': np.divide(succ_in_sum, out_deg, out=np.zeros(n), where=out_deg > 0),
        'succ_in_max': succ_in_max,
    })

### Family B — "What position does it hold?" (12 features, computed per time step)

**B1 · `pr_fwd`** — domain $[0.15,\ n_t)$, mean exactly 1 in every time step
$$PR(v) = \frac{1-\alpha}{n_t} + \alpha \sum_{u \in N^-(v)} \frac{PR(u)}{d^+(u)} + \frac{\alpha}{n_t} \sum_{u :\, d^+(u) = 0} PR(u), \qquad \alpha = 0.85, \qquad \mathtt{pr\_fwd}(v) = n_t \cdot PR(v)$$
How much money "accumulates" on $v$ from upstream: high if $v$ receives from many transactions that are themselves fed by many. 1 is the average of the time step. $\alpha = 0.85$ is the standard value; on this data the ranking of the nodes barely changes for $\alpha$ between 0.5 and 0.95 (Spearman $\rho \ge 0.96$ in every time step).

**B2 · `pr_rev`** — same domain
The same formula on the graph with reversed edges ($G_t^\top$): how much $v$ feeds what lies downstream.

**B3 · `betw`** — domain $[0, 1]$
$$\frac{1}{\binom{n_t - 1}{2}} \sum_{\{s,t\} \subseteq V_t \setminus \{v\}} \frac{\sigma_{st}(v)}{\sigma_{st}} \qquad \text{on } \hat G_t$$
$\sigma_{st}$ is the number of shortest paths between $s$ and $t$, $\sigma_{st}(v)$ the number of those that pass through $v$. How much $v$ is a **bridge**: the share of the shortest connections between other pairs of nodes that go through $v$. Exact, computed with igraph.

**B4 · `harm_in`** — domain $[0,\ n_t - 1]$
$$\sum_{u \neq v,\ \vec d(u,v) < \infty} \frac{1}{\vec d(u,v)} \;=\; \mathtt{in\_deg}(v) + \frac{\mathtt{anc2}(v)}{2} + \frac{r_3(v)}{3} + \frac{r_4(v)}{4} + \dots$$
where $r_k(v)$ is the number of ancestors at directed distance exactly $k$. All the history behind $v$, at any distance, with closer ancestors weighing more: it extends `in_deg` and `anc2` to every distance. It is **not** divided by $n_t - 1$: only ancestors count, so the value does not grow with the size of the time step by construction (dividing would make a node look less central just because its time step is large).

**B5 · `harm_out`** — domain $[0,\ n_t - 1]$
$$\sum_{w \neq v,\ \vec d(v,w) < \infty} \frac{1}{\vec d(v,w)}$$
All the "future" in front of $v$: the same count on the descendants.

**B6 · `eig`** — domain $[0, 1]$
$$\hat A\, x = \lambda_{\max}\, x, \qquad x > 0, \qquad \mathtt{eig}(v) = \frac{x_v}{\max_u x_u}$$
$x$ is unique because $\hat G_t$ is connected (Perron–Frobenius). Is $v$ linked to nodes that are themselves well linked? 1 is the most central node of the time step. The undirected version is used because on a DAG the directed one does not exist. In theory $x_v > 0$ for every node, but the score decays exponentially with the distance from the centre: half of the nodes are below $2 \cdot 10^{-6}$ and about 16% below $10^{-12}$, where the eigensolver has no precision left. Values below $10^{-12}$ are therefore set to 0.

**B7 · `hub`** — domain $[0,\ n_t)$, mean 1
$$h \propto A\, a, \quad a \propto A^\top h \quad \big(h = \text{principal eigenvector of } AA^\top,\ \textstyle\sum_v h_v = 1\big), \qquad \mathtt{hub}(v) = n_t\, h_v$$
$v$ sends money to transactions that receive from many (good "authorities"). Computed by power iteration from the uniform vector, as in the original HITS algorithm, which also makes the result unique when the top eigenvalue is repeated.

**B8 · `auth`** — domain $[0,\ n_t)$, mean 1
$$a \propto A^\top h, \quad \textstyle\sum_v a_v = 1, \qquad \mathtt{auth}(v) = n_t\, a_v$$
$v$ receives money from transactions that send to many (good "hubs").

**B9 · `core`** — domain $\{1, 2, \dots\}$
$$\max\{k : v \in k\text{-core}(\hat G_t)\}$$
The $k$-core is the largest subgraph where every node has degree at least $k$. How deep $v$ sits in a dense group of redundant links; 1 means tree-like periphery.

**B10 · `katz_in`** — domain $[0, \infty)$
$$\sum_{k \ge 1} \gamma^k \big[(A^\top)^k \mathbf 1\big]_v \;=\; \sum_{u \in N^-(v)} \gamma\,\big(1 + \mathtt{katz\_in}(u)\big), \qquad \gamma = 0.5$$
$\big[(A^\top)^k \mathbf 1\big]_v$ is the number of directed paths of length $k$ that end in $v$. How many money paths converge on $v$, each extra step weighing half. The series is finite because the graph is a DAG, and the right-hand form computes it in one pass in topological order. A few nodes in regions with many parallel paths get very large values (up to about $7 \cdot 10^{10}$, while 99% of the nodes are below 20): the log transform of the next step takes care of this.

**B11 · `katz_out`** — domain $[0, \infty)$
The same formula with $A$ instead of $A^\top$: the paths that start from $v$.

**B12 · `clo`** — domain $(0, 1]$
$$\frac{n_t - 1}{\sum_{u \in V_t \setminus \{v\}} \mathrm{dist}(u,v)} \qquad \text{on } \hat G_t$$
The inverse of the mean undirected distance from $v$ to every other node of its time step: is $v$ in the **centre** or in the **periphery** of the time step, whatever the direction of the money? Unlike B4 and B5, it also reaches the "cousins" (nodes linked to $v$ only through a common ancestor or descendant). Exact, computed with igraph.

In [8]:
def igraph_undirected(n, src, dst):
    return ig.Graph(n=n, edges=list(zip(src.tolist(), dst.tolist())), directed=False)


def pagerank_scaled(n, src, dst, alpha=PAGERANK_ALPHA, tol=1e-12, max_iter=100_000):
    """B1 / B2. PageRank by power iteration, multiplied by n (mean = 1).
    Nodes without successors give their score evenly to every node."""
    out_deg = np.bincount(src, minlength=n).astype(float)
    dangling = out_deg == 0
    p = np.full(n, 1 / n)
    for _ in range(max_iter):
        q = alpha * np.bincount(dst, weights=p[src] / out_deg[src], minlength=n)
        q += (alpha * p[dangling].sum() + 1 - alpha) / n
        if np.abs(q - p).sum() < tol:
            return n * q
        p = q
    raise RuntimeError('PageRank did not converge')


def harmonic_raw(nbrs):
    """B4 / B5. For every node, the sum of 1 / distance over the nodes it reaches along `nbrs`
    (breadth-first search; predecessors -> ancestors, successors -> descendants)."""
    out = np.zeros(len(nbrs))
    for v in range(len(nbrs)):
        dist = {v: 0}
        queue = deque([v])
        total = 0.0
        while queue:
            x = queue.popleft()
            for y in nbrs[x]:
                if y not in dist:
                    dist[y] = dist[x] + 1
                    total += 1 / dist[y]
                    queue.append(y)
        out[v] = total
    return out


def hits_scaled(n, src, dst, tol=1e-12, max_iter=100_000):
    """B7 / B8. HITS by power iteration from the uniform vector, multiplied by n (mean = 1)."""
    h = np.full(n, 1 / n)
    for _ in range(max_iter):
        a = np.bincount(dst, weights=h[src], minlength=n)
        a /= a.sum()
        h_new = np.bincount(src, weights=a[dst], minlength=n)
        h_new /= h_new.sum()
        converged = np.abs(h_new - h).sum() < tol
        h = h_new
        if converged:
            break
    else:
        warnings.warn('HITS did not fully converge: the last iterate is used')
    a = np.bincount(dst, weights=h[src], minlength=n)
    return n * h, n * a / a.sum()


def topological_order(preds, succs):
    """Kahn's algorithm: every node comes after all its predecessors."""
    missing = np.array([len(p) for p in preds])
    queue = deque(np.flatnonzero(missing == 0).tolist())
    order = []
    while queue:
        v = queue.popleft()
        order.append(v)
        for w in succs[v]:
            missing[w] -= 1
            if missing[w] == 0:
                queue.append(w)
    assert len(order) == len(preds), 'the graph has a cycle'
    return order


def katz_dag(preds, succs, gamma=KATZ_GAMMA):
    """B10 / B11. On a DAG: katz_in(v) = sum over predecessors u of gamma * (1 + katz_in(u)),
    computed in topological order (and katz_out in reverse order)."""
    order = topological_order(preds, succs)
    k_in, k_out = np.zeros(len(order)), np.zeros(len(order))
    for v in order:
        for u in preds[v]:
            k_in[v] += gamma * (1 + k_in[u])
    for v in reversed(order):
        for w in succs[v]:
            k_out[v] += gamma * (1 + k_out[w])
    return k_in, k_out


def family_B_fast(n, src, dst):
    """The ten B features that take seconds (all except betweenness and closeness)."""
    preds, succs = neighbour_lists(n, src, dst)
    g = igraph_undirected(n, src, dst)
    eig = np.abs(np.array(g.eigenvector_centrality()))
    eig /= eig.max()
    eig[eig < EIG_ZERO] = 0.0   # below the precision of the eigensolver (see B6)
    hub, auth = hits_scaled(n, src, dst)
    katz_in, katz_out = katz_dag(preds, succs)
    return pd.DataFrame({
        'pr_fwd': pagerank_scaled(n, src, dst),
        'pr_rev': pagerank_scaled(n, dst, src),
        'harm_in': harmonic_raw(preds),
        'harm_out': harmonic_raw(succs),
        'eig': eig,
        'hub': hub,
        'auth': auth,
        'core': np.array(g.coreness()),
        'katz_in': katz_in,
        'katz_out': katz_out,
    })


def betweenness(n, src, dst):
    """B3. Exact betweenness on the undirected graph, divided by the number of pairs C(n-1, 2)."""
    raw = np.array(igraph_undirected(n, src, dst).betweenness(directed=False))
    return pd.DataFrame({'betw': raw / ((n - 1) * (n - 2) / 2)})


def closeness(n, src, dst):
    """B12. Exact closeness on the undirected graph: (n - 1) / sum of the distances."""
    return pd.DataFrame({'clo': np.array(igraph_undirected(n, src, dst).closeness(normalized=True))})

### Family C — "What shape does its neighbourhood have?" (7 features)

**C1 · `is_fan_in`** — domain $\{0, 1\}$
$$\mathbb 1\big[d^-(v) \ge \tau^-\big], \qquad \tau^- = \min\big\{k \in \mathbb N : \text{share of train nodes with } d^- \ge k \le 5\%\big\}$$
$\tau^-$ is computed on the nodes of the train time steps (1–30) and then used unchanged on validation and test. It flags the top 5% at most. A plain 95th percentile would not work: most degrees are 1 or 2, so the percentile lands on a tie (it is 2, and $d^- \ge 2$ would flag 11% of the nodes, $d^+ \ge 2$ even 22%). On this data $\tau^- = \tau^+ = 3$. Does $v$ collect money from many transactions (consolidation)?

**C2 · `is_fan_out`** — domain $\{0, 1\}$
$$\mathbb 1\big[d^+(v) \ge \tau^+\big]$$
$\tau^+$ is computed in the same way. Does $v$ split money towards many transactions (distribution, smurfing)?

**C3 · `chain_len`** — domain $\mathbb N$ (0 = not in a chain)
Let $\mathcal C_t = \{v : d^-(v) = d^+(v) = 1\}$. If $v \in \mathcal C_t$, the value is the number of nodes of the connected component of $v$ in the subgraph induced by $\mathcal C_t$ (always a simple path); otherwise it is 0. Is $v$ a link of a "one input, one output" chain, like a peel chain, and how long is the chain?

**C4 · `diam_out`** — domain $\mathbb N$
$$\Big\lvert \big\{ w : \lvert \{ m : (v,m) \in E_t,\ (m,w) \in E_t \} \rvert \ge 2 \big\} \Big\rvert$$
How many times the money of $v$ splits and merges again two steps later: a scatter-gather starting from $v$.

**C5 · `diam_in`** — domain $\mathbb N$
$$\Big\lvert \big\{ u : \lvert \{ m : (u,m) \in E_t,\ (m,v) \in E_t \} \rvert \ge 2 \big\} \Big\rvert$$
How many times $v$ merges money that had split two steps before.

**C6 · `orc_mean`** — domain $[-2, 1]$ in theory
$$\frac{1}{\deg(v)} \sum_{u \in N(v)} \kappa(v,u), \qquad \kappa(x,y) = 1 - W_1(m_x, m_y)$$
The Ollivier–Ricci curvature of an edge, on $\hat G_t$, with:
- **mass of $x$:** $m_x = \alpha\, \delta_x + \frac{1 - \alpha}{\deg(x)} \sum_{z \in N(x)} \delta_z$ with $\alpha = 1/2$: half of the mass stays on $x$, the other half is split evenly among its neighbours;
- **Wasserstein distance:** $W_1(\mu, \nu) = \min_\pi \sum_{a,b} \pi(a,b)\, \mathrm{dist}(a,b)$, the cheapest way to move the mass $\mu$ until it becomes $\nu$. On an edge $\mathrm{dist}(x,y) = 1$, so the formula does not divide by it.

On average, does the neighbourhood of $v$ overlap with those of its neighbours (κ > 0: dense or star-like area) or move away from them, as in a tree or a bottleneck (κ < 0)? $W_1$ is solved **exactly** as a min-cost flow on integer masses.

**C7 · `orc_min`** — domain $[-2, 1]$
$$\min_{u \in N(v)} \kappa(v,u)$$
The most "bottleneck-like" link of $v$.

In [9]:
def fan_threshold(degree, share=FAN_SHARE):
    """C1 / C2. Smallest integer k such that at most `share` of the nodes have degree >= k."""
    k = 1
    while (degree >= k).mean() > share:
        k += 1
    return k


def chain_length(n, src, dst, in_deg, out_deg):
    """C3. Size of the chain (component of nodes with one input and one output) containing v; 0 otherwise."""
    in_chain = (in_deg == 1) & (out_deg == 1)
    chain_nodes = np.flatnonzero(in_chain)
    parent = {v: v for v in chain_nodes.tolist()}

    def root(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    keep = in_chain[src] & in_chain[dst]
    for u, v in zip(src[keep].tolist(), dst[keep].tolist()):
        parent[root(u)] = root(v)
    roots = np.array([root(v) for v in chain_nodes.tolist()], dtype=np.int64)
    _, component, size = np.unique(roots, return_inverse=True, return_counts=True)
    out = np.zeros(n, dtype=np.int64)
    out[chain_nodes] = size[component]
    return out


def diamonds(nbrs):
    """C4 / C5. Number of nodes reached from v by at least two different paths of length 2 along `nbrs`."""
    out = np.zeros(len(nbrs), dtype=np.int64)
    for v, first in enumerate(nbrs):
        paths = {}
        for m in first:
            for w in nbrs[m]:
                paths[w] = paths.get(w, 0) + 1
        out[v] = sum(1 for c in paths.values() if c >= 2)
    return out


def family_C_local(n, src, dst, tau_in, tau_out):
    """C1-C5: fan flags, chain length and diamonds."""
    in_deg = np.bincount(dst, minlength=n)
    out_deg = np.bincount(src, minlength=n)
    preds, succs = neighbour_lists(n, src, dst)
    return pd.DataFrame({
        'is_fan_in': (in_deg >= tau_in).astype(np.int64),
        'is_fan_out': (out_deg >= tau_out).astype(np.int64),
        'chain_len': chain_length(n, src, dst, in_deg, out_deg),
        'diam_out': diamonds(succs),
        'diam_in': diamonds(preds),
    })


def ollivier_ricci(n, src, dst, alpha=ORC_ALPHA, report_every=60):
    """C6 / C7. Exact Ollivier-Ricci curvature of every edge of the undirected graph,
    then mean and minimum over the edges of each node.

    The masses are scaled to integers, so W1 is solved exactly as a min-cost flow.
    Along an edge every node of the two supports is within distance 3, so the distances
    come from the neighbour sets without a full shortest-path search."""
    nbrs = [set() for _ in range(n)]
    for u, v in zip(src.tolist(), dst.tolist()):
        nbrs[u].add(v)
        nbrs[v].add(u)

    def dist(a, b):
        if a == b:
            return 0
        if b in nbrs[a]:
            return 1
        if not nbrs[a].isdisjoint(nbrs[b]):
            return 2
        return 3

    def masses(x, scale):
        # alpha of the mass on x, the rest split evenly among the neighbours (all integers)
        m = {x: alpha * scale}
        for z in nbrs[x]:
            m[z] = (1 - alpha) * scale / len(nbrs[x])
        return {node: int(mass) for node, mass in m.items()}

    curv_sum, curv_min = np.zeros(n), np.full(n, np.inf)
    last_report = time.time()
    for i, (x, y) in enumerate(zip(src.tolist(), dst.tolist())):
        scale = alpha.denominator * math.lcm(len(nbrs[x]), len(nbrs[y]))
        m_x, m_y = masses(x, scale), masses(y, scale)
        flow = nx.DiGraph()
        for a, mass in m_x.items():
            flow.add_node(('from', a), demand=-mass)
        for b, mass in m_y.items():
            flow.add_node(('to', b), demand=mass)
        for a in m_x:
            for b in m_y:
                flow.add_edge(('from', a), ('to', b), weight=dist(a, b))
        cost, _ = nx.network_simplex(flow)
        kappa = 1 - cost / scale
        for v in (x, y):
            curv_sum[v] += kappa
            curv_min[v] = min(curv_min[v], kappa)
        if time.time() - last_report > report_every:
            print(f'    ... {i + 1:,}/{len(src):,} edges of this time step', flush=True)
            last_report = time.time()
    deg = np.array([len(s) for s in nbrs])
    return pd.DataFrame({'orc_mean': curv_sum / deg, 'orc_min': curv_min})

## 2. Test on a toy graph

Before touching the real data, every feature function runs on a small graph whose values were computed independently and checked by hand. If a formula or a library call does not behave as expected, the notebook stops here.

The toy graph is a **cake recipe**: an edge u → v means "u goes into v". It behaves like Bitcoin: things flow forward, several inputs merge into one (ingredients into a dough, inputs into a transaction), one thing splits into many (a cake into slices, a transaction paying many outputs), and nothing goes back, so the graph is a DAG.

```
F (flour)  ──┐
B (butter) ──┼──────► I (dough) ──┐
U (eggs)   ──┤  U and Z go        ├──► T (cake) ──► K (baked) ──► R (cooled) ──► D (decorated) ──┬──► P1
Z (sugar)  ──┤  into I and C      │                                                              ├──► P2
L (milk)   ──┴──────► C (cream) ──┘                                                              └──► P3
```

A few values to read: `anc2(T) = 5` (two steps back the cake "remembers" all 5 ingredients); `harm_in(T) = 2 + 5/2 = 4.5`; `betw(T) = 0.564` (T is the only bridge between the top and the bottom of the recipe); `katz_in(T) = 2 × 0.5 + 7 × 0.25 = 2.75` (2 paths of length 1 and 7 of length 2); `chain_len(K) = chain_len(R) = 2` (the only nodes with one input and one output); `diam_in(T) = 2` (eggs and sugar split into I and C and merge again in T); curvature −0.25 on the bottleneck edge R–D. With only 14 nodes a percentile makes no sense, so here the C1/C2 threshold is τ = 3.

In [10]:
TOY_EDGES = [('F', 'I'), ('B', 'I'), ('U', 'I'), ('Z', 'I'), ('U', 'C'), ('Z', 'C'), ('L', 'C'),
             ('I', 'T'), ('C', 'T'), ('T', 'K'), ('K', 'R'), ('R', 'D'), ('D', 'P1'), ('D', 'P2'), ('D', 'P3')]
TOY_NODES = ['F', 'B', 'U', 'Z', 'L', 'I', 'C', 'T', 'K', 'R', 'D', 'P1', 'P2', 'P3']
position = {v: i for i, v in enumerate(TOY_NODES)}
toy_graph = (len(TOY_NODES),
             np.array([position[u] for u, _ in TOY_EDGES]),
             np.array([position[v] for _, v in TOY_EDGES]))

toy = pd.concat([family_A(*toy_graph), family_B_fast(*toy_graph), betweenness(*toy_graph),
                 closeness(*toy_graph), family_C_local(*toy_graph, tau_in=3, tau_out=3),
                 ollivier_ricci(*toy_graph)], axis=1)
toy.index = TOY_NODES

# Expected values, computed independently of this notebook (rounded to 4 decimals).
TOY_COLUMNS = ['in_deg', 'out_deg', 'anc2', 'desc2', 'pred_out_mean', 'pred_out_max', 'succ_in_mean', 'succ_in_max',
               'pr_fwd', 'pr_rev', 'betw', 'harm_in', 'harm_out', 'eig', 'hub', 'auth', 'core', 'katz_in', 'katz_out',
               'clo', 'is_fan_in', 'is_fan_out', 'chain_len', 'diam_out', 'diam_in', 'orc_mean', 'orc_min']
TOY_EXPECTED = {
    'F': [0, 1, 0, 1, 0, 0, 4, 4, 0.3093, 0.6401, 0, 0, 2.7833, 0.3559, 2.2074, 0, 1, 0, 1.0156, 0.2826, 0, 0, 0, 0, 0, 0.2, 0.2],
    'B': [0, 1, 0, 1, 0, 0, 4, 4, 0.3093, 0.6401, 0, 0, 2.7833, 0.3559, 2.2074, 0, 1, 0, 1.0156, 0.2826, 0, 0, 0, 0, 0, 0.2, 0.2],
    'U': [0, 2, 0, 1, 0, 0, 3.5, 4, 0.3093, 0.9737, 0.0256, 0, 3.7833, 0.6602, 3.9309, 0, 2, 0, 2.0312, 0.3095, 0, 0, 0, 1, 0, 0.175, 0.1],
    'Z': [0, 2, 0, 1, 0, 0, 3.5, 4, 0.3093, 0.9737, 0.0256, 0, 3.7833, 0.6602, 3.9309, 0, 2, 0, 2.0312, 0.3095, 0, 0, 0, 1, 0, 0.175, 0.1],
    'L': [0, 1, 0, 1, 0, 0, 3, 3, 0.3093, 0.7235, 0, 0, 2.7833, 0.3043, 1.7235, 0, 1, 0, 1.0156, 0.2708, 0, 0, 0, 0, 0, 0.25, 0.25],
    'I': [4, 1, 0, 1, 1.5, 2, 2, 2, 1.0979, 1.1774, 0.391, 4, 2.6833, 1, 0, 7.8617, 2, 2, 1.0312, 0.3824, 1, 0, 0, 0, 0, 0.0933, -0.1333],
    'C': [3, 1, 0, 1, 1.6667, 2, 2, 2, 0.835, 1.1774, 0.25, 3, 2.6833, 0.8549, 0, 6.1383, 2, 1.5, 1.0312, 0.3611, 1, 0, 0, 0, 0, 0.1667, -0.0833],
    'T': [2, 1, 5, 1, 1, 1, 1, 1, 1.9522, 1.853, 0.5641, 4.5, 2.5833, 0.7773, 0, 0, 2, 2.75, 1.0625, 0.4333, 0, 0, 0, 0, 2, -0.1278, -0.1667],
    'K': [1, 1, 2, 1, 1, 1, 1, 1, 1.9686, 1.7213, 0.5128, 3.6667, 2.5, 0.3291, 0, 0, 1, 1.875, 1.125, 0.4062, 0, 0, 2, 0, 0, -0.0833, -0.1667],
    'R': [1, 1, 1, 3, 1, 1, 1, 1, 1.9826, 1.5663, 0.4615, 3.4167, 2.5, 0.1472, 0, 0, 1, 1.4375, 1.25, 0.3611, 0, 0, 2, 0, 0, -0.125, -0.25],
    'D': [1, 3, 1, 0, 1, 1, 1, 1, 1.9944, 1.3841, 0.4231, 3.3333, 3, 0.0845, 0, 0, 1, 1.2188, 1.5, 0.3095, 0, 1, 0, 0, 0, 0.125, -0.25],
    'P1': [1, 0, 1, 0, 3, 3, 0, 0, 0.8743, 0.3899, 0, 3.3167, 0, 0.0301, 0, 0, 1, 1.1094, 0, 0.2407, 0, 0, 0, 0, 0, 0.25, 0.25],
}
TOY_EXPECTED['P2'] = TOY_EXPECTED['P3'] = TOY_EXPECTED['P1']
expected = pd.DataFrame.from_dict(TOY_EXPECTED, orient='index', columns=TOY_COLUMNS).loc[TOY_NODES]

assert sorted(toy.columns) == sorted(TOY_COLUMNS), 'unexpected feature names'
wrong = (toy[TOY_COLUMNS] - expected).abs() > 1e-3
assert not wrong.any().any(), f'toy test failed for: {sorted(set(wrong.columns[wrong.any()]))}'
print('Toy test passed: all 27 features match the expected values on the 14 toy nodes.')

Toy test passed: all 27 features match the expected values on the 14 toy nodes.


## 3. Load the graph and check it

The features are built on two properties of the data, checked here: there are no edges between different time steps, and every time step is exactly one connected component. The graph must also be a DAG, without self-loops, duplicate edges or reciprocal edges.

In [11]:
features = pd.read_csv(LOCAL_INPUT / 'txs_features.csv', usecols=['txId', 'Time step'])
edges = pd.read_csv(LOCAL_INPUT / 'txs_edgelist.csv')

TXID = features['txId'].to_numpy()
STEP = features['Time step'].to_numpy()
N = len(TXID)
index_of = pd.Series(np.arange(N), index=TXID)
SRC = index_of.loc[edges['txId1']].to_numpy()   # fails if an edge points to an unknown transaction
DST = index_of.loc[edges['txId2']].to_numpy()
STEPS = sorted(set(STEP.tolist()))
IN_DEG = np.bincount(DST, minlength=N)
OUT_DEG = np.bincount(SRC, minlength=N)

assert N == 203_769 and features['txId'].is_unique
assert STEPS == list(range(1, 50))
assert (SRC != DST).all(), 'self-loops'
pairs = pd.DataFrame({'u': SRC, 'v': DST})
assert not pairs.duplicated().any(), 'duplicate edges'
assert pairs.merge(pairs.rename(columns={'u': 'v', 'v': 'u'})).empty, 'reciprocal edges (u -> v and v -> u)'
assert (STEP[SRC] == STEP[DST]).all(), 'edges between different time steps'
assert ig.Graph(n=N, edges=list(zip(SRC.tolist(), DST.tolist())), directed=True).is_dag(), 'the graph has a cycle'

# One small graph per time step, with local node numbers 0 .. n_t - 1.
STEP_NODES, STEP_GRAPH = {}, {}
local = np.empty(N, dtype=np.int64)
edge_step = STEP[SRC]
for t in STEPS:
    nodes = np.flatnonzero(STEP == t)
    local[nodes] = np.arange(len(nodes))
    in_t = edge_step == t
    STEP_NODES[t] = nodes
    STEP_GRAPH[t] = (len(nodes), local[SRC[in_t]], local[DST[in_t]])
    assert igraph_undirected(*STEP_GRAPH[t]).is_connected(), f'time step {t} is not one connected component'

print(f'{N:,} transactions, {len(SRC):,} edges, {len(STEPS)} time steps '
      f'({min(len(v) for v in STEP_NODES.values()):,} to {max(len(v) for v in STEP_NODES.values()):,} nodes each).')
print('All checks passed.')

203,769 transactions, 234,355 edges, 49 time steps (1,089 to 7,880 nodes each).
All checks passed.


## 4. Family A

All eight features are local, so they are computed on the whole graph at once (it gives the same result as one time step at a time, because no edge crosses time steps). It takes seconds.

In [12]:
A_PATH = OUT_DIR / 'A_features.csv'
if A_PATH.exists():
    print('A_features.csv is already on Drive: skipped.')
else:
    start = time.time()
    A = family_A(N, SRC, DST)
    A.insert(0, 'txId', TXID)

    assert len(A) == N and not A.isna().any().any()
    assert (A.drop(columns='txId') >= 0).all().all()
    assert ((A.pred_out_mean == 0) | (A.pred_out_mean >= 1)).all() and (A.pred_out_max >= A.pred_out_mean).all()
    assert ((A.succ_in_mean == 0) | (A.succ_in_mean >= 1)).all() and (A.succ_in_max >= A.succ_in_mean).all()
    assert ((A.in_deg == 0) == (A.pred_out_max == 0)).all() and ((A.out_deg == 0) == (A.succ_in_max == 0)).all()

    write_csv(A, A_PATH)
    print(f'A_features.csv written to Drive in {fmt_time(time.time() - start)}.')

A_features.csv written to Drive in 1.8s.


## 5. Family C

C1–C5 are local and take seconds. The Ollivier–Ricci curvature (C6, C7) solves one small transport problem per edge, so it runs one time step at a time with a checkpoint after each one. The expected work of a time step is the sum, over its edges, of the sizes of the two transport problems $(\deg(x) + 1)(\deg(y) + 1)$.

In [13]:
C_PATH = OUT_DIR / 'C_features.csv'
C_LOCAL_COLUMNS = ['is_fan_in', 'is_fan_out', 'chain_len', 'diam_out', 'diam_in']

# C1-C5 take about a second, so they are always computed: if C_features.csv was written by an
# older version of this notebook, its C1-C5 columns are refreshed without recomputing C6, C7.
start = time.time()
train = np.isin(STEP, TRAIN_STEPS)
tau_in, tau_out = fan_threshold(IN_DEG[train]), fan_threshold(OUT_DEG[train])
print(f'Fan thresholds from the train time steps: tau_in = {tau_in}, tau_out = {tau_out}')
C_local = family_C_local(N, SRC, DST, tau_in, tau_out)
C_local.insert(0, 'txId', TXID)
print(f'C1-C5 done in {fmt_time(time.time() - start)} '
      f'({C_local.is_fan_in.mean():.1%} of the nodes are fan-in, {C_local.is_fan_out.mean():.1%} fan-out).')

if C_PATH.exists():
    C = pd.read_csv(C_PATH)
    assert len(C) == N and (C['txId'].to_numpy() == TXID).all(), 'C_features.csv: rows missing or in the wrong order'
    if (C[C_LOCAL_COLUMNS].to_numpy() == C_local[C_LOCAL_COLUMNS].to_numpy()).all():
        print('C_features.csv is already on Drive: skipped.')
    else:
        C[C_LOCAL_COLUMNS] = C_local[C_LOCAL_COLUMNS].to_numpy()
        write_csv(C, C_PATH)
        print('C_features.csv was written by an older version: C1-C5 refreshed, C6-C7 kept.')
else:
    degree = IN_DEG + OUT_DEG
    edge_work = (degree[SRC] + 1) * (degree[DST] + 1)
    orc_work = {t: int(edge_work[edge_step == t].sum()) for t in STEPS}
    C_orc = run_per_step('C_ollivier_ricci', ollivier_ricci, orc_work)

    C = in_node_order(C_local.merge(C_orc, on='txId', validate='one_to_one'))
    assert C.is_fan_in.isin([0, 1]).all() and C.is_fan_out.isin([0, 1]).all()
    assert ((C.chain_len > 0) == ((IN_DEG == 1) & (OUT_DEG == 1))).all()
    assert (C.diam_out >= 0).all() and (C.diam_in >= 0).all()
    assert C.orc_mean.between(-2, 1).all() and (C.orc_min <= C.orc_mean + 1e-12).all()

    write_csv(C, C_PATH)
    shutil.rmtree(PARTIAL_DIR / 'C_ollivier_ricci')
    print(f'C_features.csv written to Drive in {fmt_time(time.time() - start)}.')

Fan thresholds from the train time steps: tau_in = 3, tau_out = 3
C1-C5 done in 1.0s (4.2% of the nodes are fan-in, 3.2% fan-out).
[C_ollivier_ricci] 0 time steps already on Drive, 49 to compute.
[C_ollivier_ricci] time step  1 (7,880 nodes) in 20.2s | 1/49 steps, 7,880/203,769 nodes, 4.4% of the work | elapsed 20.2s, left ~7m 15s
[C_ollivier_ricci] time step  2 (4,544 nodes) in 5.1s | 2/49 steps, 12,424/203,769 nodes, 5.5% of the work | elapsed 25.4s, left ~7m 13s
[C_ollivier_ricci] time step  3 (6,621 nodes) in 6.5s | 3/49 steps, 19,045/203,769 nodes, 7.9% of the work | elapsed 31.9s, left ~6m 13s
[C_ollivier_ricci] time step  4 (5,693 nodes) in 16.1s | 4/49 steps, 24,738/203,769 nodes, 13.1% of the work | elapsed 48.0s, left ~5m 19s
[C_ollivier_ricci] time step  5 (6,803 nodes) in 13.0s | 5/49 steps, 31,541/203,769 nodes, 17.2% of the work | elapsed 1m 00s, left ~4m 53s
[C_ollivier_ricci] time step  6 (4,328 nodes) in 5.0s | 6/49 steps, 35,869/203,769 nodes, 18.9% of the work | elap

## 6. Family B

The ten fast features take seconds per time step. Betweenness (B3) and closeness (B12) need one breadth-first search from every node, so their cost grows like $n_t \cdot m_t$ (nodes × edges of the time step): they run one time step at a time with a checkpoint after each one, and the estimate of the remaining time is weighted by $n_t \cdot m_t$.

In [14]:
B_PATH = OUT_DIR / 'B_features.csv'
if B_PATH.exists():
    print('B_features.csv is already on Drive: skipped.')
else:
    start = time.time()
    parts = []
    progress = Progress('B_fast', {t: STEP_GRAPH[t][0] + len(STEP_GRAPH[t][1]) for t in STEPS})
    for t in STEPS:
        step_start = time.time()
        part = family_B_fast(*STEP_GRAPH[t])
        part.insert(0, 'txId', TXID[STEP_NODES[t]])
        parts.append(part)
        progress.step(t, time.time() - step_start)
    B_fast = pd.concat(parts, ignore_index=True)

    bfs_work = {t: STEP_GRAPH[t][0] * len(STEP_GRAPH[t][1]) for t in STEPS}
    B_betw = run_per_step('B_betweenness', betweenness, bfs_work)
    B_clo = run_per_step('B_closeness', closeness, bfs_work)

    B = B_fast.merge(B_betw, on='txId', validate='one_to_one').merge(B_clo, on='txId', validate='one_to_one')
    B = in_node_order(B)[['txId', 'pr_fwd', 'pr_rev', 'betw', 'harm_in', 'harm_out', 'eig',
                          'hub', 'auth', 'core', 'katz_in', 'katz_out', 'clo']]

    by_step = B.groupby(STEP)
    n_t = pd.Series(STEP).map(lambda t: len(STEP_NODES[t])).to_numpy()
    for col in ['pr_fwd', 'pr_rev', 'hub', 'auth']:
        assert np.allclose(by_step[col].mean(), 1, atol=1e-6), f'{col}: mean is not 1 in every time step'
    assert (B.pr_fwd >= 1 - PAGERANK_ALPHA - 1e-9).all() and (B.pr_rev >= 1 - PAGERANK_ALPHA - 1e-9).all()
    assert B.betw.between(0, 1).all() and B.clo.between(0, 1).all() and (B.clo > 0).all()
    assert B.eig.between(0, 1).all() and np.allclose(by_step['eig'].max(), 1)
    assert ((B.harm_in == 0) == (IN_DEG == 0)).all() and ((B.harm_out == 0) == (OUT_DEG == 0)).all()
    assert (B.harm_in <= n_t - 1).all() and (B.harm_out <= n_t - 1).all()
    assert ((B.katz_in == 0) == (IN_DEG == 0)).all() and ((B.katz_out == 0) == (OUT_DEG == 0)).all()
    assert np.isfinite(B[['katz_in', 'katz_out']]).all().all()
    assert (B.core >= 1).all() and (B.hub >= 0).all() and (B.auth >= 0).all()

    write_csv(B, B_PATH)
    shutil.rmtree(PARTIAL_DIR / 'B_betweenness')
    shutil.rmtree(PARTIAL_DIR / 'B_closeness')
    print(f'B_features.csv written to Drive in {fmt_time(time.time() - start)}.')

[B_fast] time step  1 (7,880 nodes) in 0.2s | 1/49 steps, 7,880/203,769 nodes, 3.9% of the work | elapsed 0.2s, left ~5.9s


/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))


[B_fast] time step  2 (4,544 nodes) in 0.2s | 2/49 steps, 12,424/203,769 nodes, 6.1% of the work | elapsed 0.4s, left ~6.9s


/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))
/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))


[B_fast] time step  3 (6,621 nodes) in 1.2s | 3/49 steps, 19,045/203,769 nodes, 9.5% of the work | elapsed 1.7s, left ~16.0s


/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))


... (41 outputs left out)

/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))
/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))


[B_fast] time step 47 (5,121 nodes) in 0.2s | 47/49 steps, 198,361/203,769 nodes, 97.4% of the work | elapsed 10.7s, left ~0.3s
[B_fast] time step 48 (2,954 nodes) in 0.1s | 48/49 steps, 201,315/203,769 nodes, 98.8% of the work | elapsed 10.8s, left ~0.1s
[B_fast] time step 49 (2,454 nodes) in 0.1s | 49/49 steps, 203,769/203,769 nodes, 100.0% of the work | elapsed 10.9s, left ~0.0s
[B_betweenness] 0 time steps already on Drive, 49 to compute.


/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))
/tmp/ipykernel_6945/1190178300.py:91: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:102
  eig = np.abs(np.array(g.eigenvector_centrality()))


[B_betweenness] time step  1 (7,880 nodes) in 4.2s | 1/49 steps, 7,880/203,769 nodes, 6.4% of the work | elapsed 4.2s, left ~1m 01s
[B_betweenness] time step  2 (4,544 nodes) in 1.2s | 2/49 steps, 12,424/203,769 nodes, 8.5% of the work | elapsed 5.5s, left ~58.8s
[B_betweenness] time step  3 (6,621 nodes) in 3.8s | 3/49 steps, 19,045/203,769 nodes, 13.3% of the work | elapsed 9.2s, left ~59.8s
[B_betweenness] time step  4 (5,693 nodes) in 3.5s | 4/49 steps, 24,738/203,769 nodes, 17.5% of the work | elapsed 12.7s, left ~1m 00s
[B_betweenness] time step  5 (6,803 nodes) in 3.3s | 5/49 steps, 31,541/203,769 nodes, 22.6% of the work | elapsed 16.0s, left ~54.8s
[B_betweenness] time step  6 (4,328 nodes) in 1.1s | 6/49 steps, 35,869/203,769 nodes, 24.6% of the work | elapsed 17.2s, left ~52.5s
[B_betweenness] time step  7 (6,048 nodes) in 2.5s | 7/49 steps, 41,917/203,769 nodes, 28.5% of the work | elapsed 19.7s, left ~49.4s
[B_betweenness] time step  8 (4,457 nodes) in 1.2s | 8/49 steps, 4

## 7. Final check and Drive sync

Reads the three files back from Drive and checks that they are complete. The last cell makes sure that every file has been uploaded to Drive; after it, Drive is disconnected (to run other cells again, run the Drive cell of section 0 first).

In [15]:
EXPECTED_COLUMNS = {
    'A_features.csv': ['in_deg', 'out_deg', 'anc2', 'desc2', 'pred_out_mean', 'pred_out_max', 'succ_in_mean', 'succ_in_max'],
    'B_features.csv': ['pr_fwd', 'pr_rev', 'betw', 'harm_in', 'harm_out', 'eig', 'hub', 'auth', 'core', 'katz_in', 'katz_out', 'clo'],
    'C_features.csv': ['is_fan_in', 'is_fan_out', 'chain_len', 'diam_out', 'diam_in', 'orc_mean', 'orc_min'],
}
for name, columns in EXPECTED_COLUMNS.items():
    df = pd.read_csv(OUT_DIR / name)
    assert list(df.columns) == ['txId'] + columns, f'{name}: unexpected columns'
    assert len(df) == N and (df['txId'].to_numpy() == TXID).all(), f'{name}: rows missing or in the wrong order'
    assert not df.isna().any().any(), f'{name}: missing values'
    print(f'{name}: {len(df):,} rows x {len(columns)} features, {(OUT_DIR / name).stat().st_size / 1e6:.1f} MB')
assert not PARTIAL_DIR.exists() or not any(PARTIAL_DIR.iterdir()), 'some checkpoints were not cleaned up'
print('All three feature files are complete.')

A_features.csv: 203,769 rows x 8 features, 5.5 MB
B_features.csv: 203,769 rows x 12 features, 24.7 MB
C_features.csv: 203,769 rows x 7 features, 7.9 MB
All three feature files are complete.


In [16]:
drive.flush_and_unmount()
print('Every file is on Drive in', OUT_DIR)

Every file is on Drive in /content/drive/MyDrive/UNIVERSITA/block-crypto/data/01-graph-feature-extractions
